# From an insurer PDF to structured data, with a Pydantic AI agent
**PDF → text → Pydantic model → AI agent fills it → structured output**

The example is Anjali Sharma's settlement letter from Arogya: no claim number, and her name is misspelt.
Set `GEMINI_API_KEY` before starting Jupyter (or paste the key when asked in step 3).

## 1. Read the PDF
First we make a sample insurer letter (using the demo's insurer styles), then pull out its text. No AI yet.

In [ ]:
import sys
from pathlib import Path
from types import SimpleNamespace
from pypdf import PdfReader

sys.path.insert(0, str(Path.cwd().parent / "demo" / "insurers"))   # the insurers' letter styles (demo code)
import documents

anjali = SimpleNamespace(patient_name="Anjali Sharma", admission_date="2026-09-11", amount_claimed=311600,
                         paid=233200, reason="Investigation charges disallowed", paid_on="2026-10-15")
Path("arogya_letter.pdf").write_bytes(documents.arogya(anjali))      # a PDF exactly like the hospital receives

text = "\n".join(page.extract_text() for page in PdfReader("arogya_letter.pdf").pages)
print(text)

## 2. The Pydantic model: the shape we want
The same fields for every insurer, whatever their letter looks like. The `description`s tell the AI what each field means.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

class Settlement(BaseModel):
    insurer: str = Field(description="The insurance company or TPA that wrote the letter")
    claim_no: str | None = Field(description="The HOSPITAL's claim number, like HSP-1011, if the letter shows one, "
                                             "else null. Not the insurer's own reference number.")
    patient_name: str = Field(description="The patient's name exactly as written, typos included")
    billed: float = Field(description="Amount the hospital billed, in rupees, as a plain number")
    paid: float = Field(description="Amount the insurer approved / paid, in rupees, as a plain number")
    reason: str = Field(description="The insurer's reason for paying less, word for word; '' if paid in full")
    category: Literal["ROOM_LIMIT", "NOT_COVERED_ITEMS", "PATIENT_SHARE", "OTHER"] = Field(
        description="ROOM_LIMIT: room above the policy limit. NOT_COVERED_ITEMS: only consumables like gloves, masks. "
                    "PATIENT_SHARE: co-pay. OTHER: anything else (tests, implants, treatment) or nothing cut.")

print(list(Settlement.model_fields))

## 3. The AI agent (Pydantic AI)
An agent = a model (Gemini) + instructions + the **output type**. Because the output type is `Settlement`, the agent must answer with a valid `Settlement`. This is how ClaimTrail's reader agent works (`integration/claimtrail/ai.py`).

In [ ]:
import os
from getpass import getpass
from pydantic_ai import Agent, NativeOutput

os.environ["PYDANTIC_AI_NO_BANNER"] = "1"
os.environ.setdefault("GOOGLE_API_KEY", os.environ.pop("GEMINI_API_KEY", "") or getpass("Gemini API key: "))

agent = Agent(
    "google:gemini-3.1-flash-lite",
    output_type=NativeOutput(Settlement),   # Gemini's native JSON output, the same as the project's reader
    system_prompt="You read insurance settlement letters sent to a hospital and extract the settlement details.",
)
agent

## 4. The agent reads the letter and fills in the details
The result is a real `Settlement` object, already checked by Pydantic (numbers are numbers, the category is one of the four).

In [ ]:
result = await agent.run(text)
settlement = result.output
settlement

## 5. The structured details
Clean data, ready for the hospital system. The money decision is plain code, not AI.

In [ ]:
import pandas as pd

print(settlement.model_dump_json(indent=2))

cut = settlement.billed - settlement.paid
decision = "Fair cut" if settlement.category in {"ROOM_LIMIT", "NOT_COVERED_ITEMS", "PATIENT_SHARE"} else "Unfair cut"
pd.DataFrame([{**settlement.model_dump(), "cut": cut, "decision": decision if cut > 100 else "Paid in full"}])